In [1]:
import pandas as pd
import requests
import time
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
import yfinance as yf
import certifi, os
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# ── Config ───────────────────────────────────────────────────
load_dotenv()
api_key = os.getenv("AV_API_KEY")
assert api_key and api_key != "True", f"Bad API key loaded: {api_key!r}"
print(f"Loaded key ending in ...{api_key[-4:]}")
DATA_DIR     = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

# ── S&P 500 universe (pulled live, not hand-typed) ───────────────────────
# N_PER_SECTOR takes a stratified random sample of that many tickers per
# GICS sector, so the ticker universe is guaranteed to be real S&P 500
# members while keeping the AV + news-scraping runtime manageable.
# Set N_PER_SECTOR = None to pull EVERY current S&P 500 constituent
# (~500 tickers) — only do this if your AV plan / news pipeline can absorb
# that volume; a full run is a very different order of magnitude of calls.
N_PER_SECTOR = 5
RANDOM_SEED = 42

SP500_WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# GICS "Information Technology" gets split into "semi" vs "tech" (by
# sub-industry) since chip names behave differently from software/hardware
# names; every other GICS sector maps 1:1 onto a sector_group bucket.
GICS_SECTOR_TO_GROUP = {
    "Information Technology": "tech",
    "Communication Services": "communication",
    "Consumer Discretionary": "consumer_discretionary",
    "Consumer Staples": "consumer_staples",
    "Financials": "financials",
    "Health Care": "healthcare",
    "Industrials": "industrials",
    "Energy": "energy",
    "Materials": "materials",
    "Utilities": "utilities",
    "Real Estate": "real_estate",
}


import ssl
ssl._create_default_https_context = ssl._create_unverified_context  # fallback only if step above still fails — see note below


def fetch_sp500_universe(n_per_sector=N_PER_SECTOR, seed=RANDOM_SEED):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/124.0.0.0 Safari/537.36"
    }
    resp = requests.get(SP500_WIKI_URL, headers=headers, timeout=10)
    resp.raise_for_status()
    tables = pd.read_html(resp.text)
    sp500 = tables[0][["Symbol", "GICS Sector", "GICS Sub-Industry"]].copy()

    sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)

    def map_sector_group(row):
        if "Semiconductor" in row["GICS Sub-Industry"]:
            return "semi"
        return GICS_SECTOR_TO_GROUP.get(
            row["GICS Sector"], row["GICS Sector"].lower().replace(" ", "_")
        )

    sp500["sector_group"] = sp500.apply(map_sector_group, axis=1)

    if n_per_sector is not None:
        sp500 = (
            sp500.groupby("sector_group", group_keys=False)[sp500.columns]
            .apply(lambda g: g.sample(min(len(g), n_per_sector), random_state=seed))
        )

    tickers = sp500["Symbol"].tolist()
    ticker_groups = dict(zip(sp500["Symbol"], sp500["sector_group"]))
    return tickers, ticker_groups

TICKERS, TICKER_GROUPS = fetch_sp500_universe()
print(
    f"Pulled {len(TICKERS)} current S&P 500 tickers across "
    f"{len(set(TICKER_GROUPS.values()))} sectors "
    f"(N_PER_SECTOR={N_PER_SECTOR})."
)

Loaded key ending in ...LPEM
Pulled 60 current S&P 500 tickers across 12 sectors (N_PER_SECTOR=5).


/var/folders/wx/dn58vfts0sx56qhtt2g77frm0000gn/T/ipykernel_4064/3741313928.py:62: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)


In [2]:
def fetch_alpha_vantage_earnings(tickers, api_key):
    """
    Queries Alpha Vantage for historical actual vs estimated EPS data.
    """
    all_records = []
    failed_tickers = []

    for ticker in tickers:
        print(f"Fetching historical earnings from Alpha Vantage for: {ticker}...")

        # Alpha Vantage Fundamental Earnings Endpoint
        url = f"https://www.alphavantage.co/query?function=EARNINGS&symbol={ticker}&apikey={api_key}"

        got_data = False
        for attempt in range(3):
            try:
                response = requests.get(url)

                if response.status_code != 200:
                    print(f"  -> Error HTTP {response.status_code}")
                    break

                data = response.json()

                # AV signals rate-limit/quota exhaustion via "Note" (older
                # free-tier message) or "Information" (current message) —
                # both mean "no data was actually returned", so retry the
                # same ticker after backing off instead of silently moving
                # on and treating the throttle response as "ticker has no
                # earnings data".
                if "Note" in data or "Information" in data:
                    msg = data.get("Note") or data.get("Information")
                    print(f"  -> Rate limit/quota hit (attempt {attempt + 1}/3): {msg}")
                    time.sleep(30)
                    continue

                if "quarterlyEarnings" in data:
                    df = pd.DataFrame(data["quarterlyEarnings"])
                    df['ticker'] = ticker
                    all_records.append(df)
                    print(f"  -> Successfully retrieved {len(df)} quarters of data.")
                    got_data = True
                else:
                    print(f"  -> No data found for {ticker}. Response keys: {list(data.keys())}")
                break

            except Exception as e:
                print(f"  -> Exception occurred: {e}")
                break

        if not got_data:
            failed_tickers.append(ticker)

        # Premium tier: no more 5 requests/min cap, so this only needs to
        # stay comfortably under the plan's per-minute ceiling (matches the
        # 1s spacing already used in news_collection.ipynb).
        time.sleep(1)

    if failed_tickers:
        print(f"\nWARNING: no data retrieved for {len(failed_tickers)} ticker(s): {failed_tickers}")

    if not all_records:
        return pd.DataFrame(), failed_tickers

    # Combine everything together
    full_df = pd.concat(all_records, ignore_index=True)

    # Keep and clean relevant columns
    # AV returns: fiscalDateEnding, reportedDate, reportedEPS, estimatedEPS, surprise, surprisePercentage
    columns_map = {
        'ticker': 'ticker',
        'reportedDate': 'earnings_date',
        'reportedEPS': 'actual_eps',
        'estimatedEPS': 'consensus_eps',
        'fiscalDateEnding': 'fiscal_date_ending'
    }

    full_df = full_df.rename(columns=columns_map)
    available_cols = [c for c in columns_map.values() if c in full_df.columns]
    full_df = full_df[available_cols]

    # Clean and force numeric formatting
    full_df['earnings_date'] = pd.to_datetime(full_df['earnings_date']).dt.date
    full_df['actual_eps'] = pd.to_numeric(full_df['actual_eps'], errors='coerce')
    full_df['consensus_eps'] = pd.to_numeric(full_df['consensus_eps'], errors='coerce')

    # Drop rows missing crucial modeling variables
    full_df = full_df.dropna(subset=['actual_eps', 'consensus_eps'])

    # Filter for the last 5 years (2021 onward)
    five_years_ago = datetime.today().date() - pd.Timedelta(days=5*365)
    full_df = full_df[full_df['earnings_date'] >= five_years_ago]

    # Group tracking alignment
    full_df['sector_group'] = full_df['ticker'].map(TICKER_GROUPS)

    # Calculate target labels (1 = Beat/Met, 0 = Miss)
    full_df['surprise_amount'] = full_df['actual_eps'] - full_df['consensus_eps']
    full_df['target_label'] = full_df['surprise_amount'].apply(lambda x: 1 if x >= 0 else 0)

    return full_df.sort_values(by=['ticker', 'earnings_date'], ascending=[True, False]).reset_index(drop=True), failed_tickers

if __name__ == "__main__":

    earnings_df, failed_tickers = fetch_alpha_vantage_earnings(TICKERS, api_key)

    if not earnings_df.empty:
        print(f"\nSuccess! Collected {len(earnings_df)} clean historical data rows.")
        print(earnings_df.head(10))

        output_path = os.path.join(DATA_DIR, "earnings_base.csv")
        if failed_tickers:
            print(f"\nNOT overwriting {output_path} — missing data for {failed_tickers}. "
                  f"Re-run this cell to retry just the failures once the API is responsive again.")
        else:
            earnings_df.to_csv(output_path, index=False)
            print(f"\nSaved master target dataframe to: '{output_path}'")
    else:
        print("\nCould not retrieve historical data. Ensure your API key is active.")

Fetching historical earnings from Alpha Vantage for: PSKY...
  -> Successfully retrieved 113 quarters of data.
Fetching historical earnings from Alpha Vantage for: LYV...
  -> Successfully retrieved 82 quarters of data.
Fetching historical earnings from Alpha Vantage for: GOOGL...
  -> Successfully retrieved 88 quarters of data.
Fetching historical earnings from Alpha Vantage for: FOX...
  -> Successfully retrieved 38 quarters of data.
Fetching historical earnings from Alpha Vantage for: TTWO...
  -> Successfully retrieved 111 quarters of data.
Fetching historical earnings from Alpha Vantage for: MAR...
  -> Successfully retrieved 120 quarters of data.
Fetching historical earnings from Alpha Vantage for: TPR...
  -> Successfully retrieved 103 quarters of data.
Fetching historical earnings from Alpha Vantage for: LULU...
  -> Successfully retrieved 76 quarters of data.
Fetching historical earnings from Alpha Vantage for: ULTA...
  -> Successfully retrieved 75 quarters of data.
Fetching 

In [3]:
# ── Config ─────────────────────────────────────────────
DATA_DIR = "../data/raw"
BASE_CSV = os.path.join(DATA_DIR, "earnings_base.csv")
PRICE_DIR = os.path.join(DATA_DIR, "ticker_prices")
os.makedirs(PRICE_DIR, exist_ok=True)

WINDOW_BEFORE = 30  # Number of days to pull before the earnings announcement

def download_market_features(base_csv_path):
    """
    Reads the base earnings file and downloads historical stock prices
    spanning the required lookback windows for each ticker.
    """
    if not os.path.exists(base_csv_path):
        print(f"Error: Base file not found at {base_csv_path}. Please run your Alpha Vantage script first.")
        return
        
    # 1. Load your earnings data
    df_earnings = pd.read_csv(base_csv_path)
    df_earnings['earnings_date'] = pd.to_datetime(df_earnings['earnings_date'])
    
    # Extract the unique list of tickers we need data for
    unique_tickers = df_earnings['ticker'].unique()
    print(f"Found {len(unique_tickers)} tickers in your base CSV file.")
    
    # 2. Find the absolute minimum and maximum date ranges needed across all stocks
    # This prevents hitting yfinance thousands of times in a slow loop
    global_start = df_earnings['earnings_date'].min() - timedelta(days=WINDOW_BEFORE + 5)
    global_end = df_earnings['earnings_date'].max() + timedelta(days=5)
    
    print(f"Downloading historical blocks from {global_start.date()} to {global_end.date()}...")
    
    # 3. Download data ticker by ticker
    for ticker in unique_tickers:
        print(f"Downloading historical price framework for: {ticker}...")
        
        try:
            # Fetch data from Yahoo Finance
            # yfinance automatically handles adjustments for stock splits/dividends out of the box
            stock_data = yf.download(ticker, start=global_start, end=global_end, progress=False)
            
            if stock_data.empty:
                print(f"  -> Warning: No market pricing returned for {ticker}")
                continue
                
            # Reset index to turn the Date from an index into a standard column
            stock_data = stock_data.reset_index()
            
            # Clean up the column naming convention (sometimes yfinance yields MultiIndex columns)
            if isinstance(stock_data.columns, pd.MultiIndex):
                stock_data.columns = stock_data.columns.droplevel(1)
                
            stock_data.columns = [col.lower() for col in stock_data.columns]
            
            # Save the raw time series block to its own file
            ticker_file = os.path.join(PRICE_DIR, f"{ticker}_daily_prices.csv")
            stock_data.to_csv(ticker_file, index=False)
            print(f"  -> Saved {len(stock_data)} days of price data to '{ticker_file}'")
            
        except Exception as e:
            print(f"  -> Failed to pull market metrics for {ticker}: {e}")

if __name__ == "__main__":
    download_market_features(BASE_CSV)
    print("\nMarket data gathering phase complete!")

Found 60 tickers in your base CSV file.
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/ABT_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/ADI_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/AFL_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/AIZ_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/AMD_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/AMP_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/APA_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/APD_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/BAX_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/BKR_daily_prices.csv'
  -> Saved 1278 days of price data to '../data/raw/ticker_prices/CL_daily_prices.csv'
  ->